## GOAL OF NOTEBOOK 03

We will:

1)clean tweets 
2)remove noise
3)normalize text
4)create RAG documents
5)prepare embedding-ready chunks

This notebook is VERY important

In [10]:
# Imports
import pandas as pd
import re


In [11]:
# Load Dataset Again
airline_df = pd.read_csv(
    "../data/raw/Twitter US Airline Sentiment/Tweets.csv"
)

airline_df = airline_df[
    [
        "airline",
        "text",
        "airline_sentiment",
        "negativereason"
    ]
]

airline_df.head()

,airline,text,airline_sentiment,negativereason
0,Virgin America,@VirginAmerica What @dhepburn said.,neutral,NaN
1,Virgin America,@VirginAmerica plus you've added commercials t...,positive,NaN
2,Virgin America,@VirginAmerica I didn't today... Must mean I n...,neutral,NaN
3,Virgin America,@VirginAmerica it's really aggressive to blast...,negative,Bad Flight
4,Virgin America,@VirginAmerica and it's a really big bad thing...,negative,Can't Tell


### WHY LOAD AGAIN?

Each notebook should be:
✅ independent
✅ reproducible
✅ runnable from scratch

This is professional notebook design.

In [13]:
# Create Text Cleaning Function
def clean_tweet(text):

    # lowercase
    text = text.lower()

    # remove urls
    text = re.sub(r"http\S+", "", text)

    # remove mentions
    text = re.sub(r"@\w+", "", text)

    # remove hashtags symbol only
    text = re.sub(r"#", "", text)

    # remove special characters
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)

    # remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

### WHY THIS IS IMPORTANT

Tweets are noisy.

They contain:

* hashtags
* mentions
* links
* emojis
* symbols

RAG systems work MUCH better on cleaned text.

In [14]:
# Apply Cleaning
airline_df["clean_text"] = airline_df["text"].apply(clean_tweet)

airline_df.head()

,airline,text,airline_sentiment,negativereason,clean_text
0,Virgin America,@VirginAmerica What @dhepburn said.,neutral,NaN,what said
1,Virgin America,@VirginAmerica plus you've added commercials t...,positive,NaN,plus you ve added commercials to the experienc...
2,Virgin America,@VirginAmerica I didn't today... Must mean I n...,neutral,NaN,i didn t today must mean i need to take anothe...
3,Virgin America,@VirginAmerica it's really aggressive to blast...,negative,Bad Flight,it s really aggressive to blast obnoxious ente...
4,Virgin America,@VirginAmerica and it's a really big bad thing...,negative,Can't Tell,and it s a really big bad thing about it


In [15]:
# View Sample Cleaned Tweets
for i in range(5):
    print("ORIGINAL:")
    print(airline_df["text"].iloc[i])
    
    print("\nCLEANED:")
    print(airline_df["clean_text"].iloc[i])
    
    print("\n" + "="*80 + "\n")

ORIGINAL:
@VirginAmerica What @dhepburn said.

CLEANED:
what said


ORIGINAL:
@VirginAmerica plus you've added commercials to the experience... tacky.

CLEANED:
plus you ve added commercials to the experience tacky


ORIGINAL:
@VirginAmerica I didn't today... Must mean I need to take another trip!

CLEANED:
i didn t today must mean i need to take another trip


ORIGINAL:
@VirginAmerica it's really aggressive to blast obnoxious "entertainment" in your guests' faces &amp; they have little recourse

CLEANED:
it s really aggressive to blast obnoxious entertainment in your guests faces amp they have little recourse


ORIGINAL:
@VirginAmerica and it's a really big bad thing about it

CLEANED:
and it s a really big bad thing about it




In [16]:
# Remove Empty Cleaned Text
airline_df = airline_df[
    airline_df["clean_text"].str.len() > 10
]

print("Empty/short tweets removed!")

Empty/short tweets removed!


We are converting raw data into:

### semantic knowledge documents

In [17]:
# Create RAG Documents
airline_df["negativereason"] = airline_df["negativereason"].fillna("No Issue")

airline_df["rag_document"] = (
    "Airline: " + airline_df["airline"].astype(str)
    + "\nSentiment: " + airline_df["airline_sentiment"].astype(str)
    + "\nIssue: " + airline_df["negativereason"].astype(str)
    + "\nCustomer Message: " + airline_df["clean_text"].astype(str)
)

In [18]:
# View RAG Documents
airline_df["rag_document"].iloc[0]

'Airline: Virgin America\nSentiment: positive\nIssue: No Issue\nCustomer Message: plus you ve added commercials to the experience tacky'

In [19]:
# Save Processed Dataset
airline_df.to_csv(
    "../data/processed/cleaned_airline_support.csv",
    index=False
)

print("Processed dataset saved successfully!")

Processed dataset saved successfully!
